# Transportation Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.7

This notebook solves the transportation problem from book section `2.7` with AMPL from Python using `amplpy`.


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def create_ampl_instance(solver: str = "highs"):
    return ampl_notebook(modules=[solver], license_uuid="default")


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Problem: Transportation

The book model minimizes total weekly shipping cost while satisfying all market demands without exceeding origin capacities.


In [3]:
ORIGIN_DATA = ["cali_plant", "medellin_plant", "bogota_warehouse"]
DESTINATION_DATA = ["pasto", "medellin", "cartagena", "cali", "manizales", "bogota"]

SUPPLY = {
    "cali_plant": 600,
    "medellin_plant": 600,
    "bogota_warehouse": 250,
}

DEMAND = {
    "pasto": 280,
    "medellin": 250,
    "cartagena": 240,
    "cali": 170,
    "manizales": 210,
    "bogota": 300,
}

COST = {
    ("cali_plant", "pasto"): 13150,
    ("cali_plant", "medellin"): 9650,
    ("cali_plant", "cartagena"): 19950,
    ("cali_plant", "cali"): 5000,
    ("cali_plant", "manizales"): 9650,
    ("cali_plant", "bogota"): 15950,
    ("medellin_plant", "pasto"): 19950,
    ("medellin_plant", "medellin"): 6500,
    ("medellin_plant", "cartagena"): 19950,
    ("medellin_plant", "cali"): 9650,
    ("medellin_plant", "manizales"): 11150,
    ("medellin_plant", "bogota"): 15950,
    ("bogota_warehouse", "pasto"): 17150,
    ("bogota_warehouse", "medellin"): 15950,
    ("bogota_warehouse", "cartagena"): 19950,
    ("bogota_warehouse", "cali"): 15950,
    ("bogota_warehouse", "manizales"): 15950,
    ("bogota_warehouse", "bogota"): 7000,
}

EXPECTED_COST = 15609000
EXPECTED_POSITIVE_ARCS = {
    ("cali_plant", "pasto"): 280,
    ("cali_plant", "cali"): 170,
    ("cali_plant", "manizales"): 150,
    ("medellin_plant", "medellin"): 250,
    ("medellin_plant", "cartagena"): 240,
    ("medellin_plant", "manizales"): 60,
    ("medellin_plant", "bogota"): 50,
    ("bogota_warehouse", "bogota"): 250,
}

assert sum(SUPPLY.values()) == sum(DEMAND.values()) == 1450


In [4]:
@timed
def solve_transport_with_ampl(origins, destinations, supply, demand, cost, solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set ORIGINS ordered;
        set DESTINATIONS ordered;

        param supply {ORIGINS} >= 0;
        param demand {DESTINATIONS} >= 0;
        param cost {ORIGINS, DESTINATIONS} >= 0;

        var Ship {o in ORIGINS, d in DESTINATIONS} integer >= 0;

        minimize TotalCost:
            sum {o in ORIGINS, d in DESTINATIONS} cost[o, d] * Ship[o, d];

        subject to SupplyLimit {o in ORIGINS}:
            sum {d in DESTINATIONS} Ship[o, d] <= supply[o];

        subject to DemandCover {d in DESTINATIONS}:
            sum {o in ORIGINS} Ship[o, d] >= demand[d];
        '''
    )
    ampl.set["ORIGINS"] = origins
    ampl.set["DESTINATIONS"] = destinations
    ampl.param["supply"] = supply
    ampl.param["demand"] = demand
    ampl.param["cost"] = cost
    ampl.solve(solver=solver)

    values = ampl.get_variable("Ship").get_values().to_dict()
    positive = {
        arc: int(round(value))
        for arc, value in values.items()
        if int(round(value)) > 0
    }
    return {
        "shipments": positive,
        "total_cost": int(round(ampl.get_objective("TotalCost").value())),
    }


In [5]:
ampl_result, ampl_time = solve_transport_with_ampl(
    origins=ORIGIN_DATA,
    destinations=DESTINATION_DATA,
    supply=SUPPLY,
    demand=DEMAND,
    cost=COST,
)

print("=== TRANSPORTATION RESULTS WITH AMPL ===")
print(f"Balanced problem check -> supply = {sum(SUPPLY.values())}, demand = {sum(DEMAND.values())}")
print(f"Solution               -> {ampl_result}")
print(f"Time                   -> {ampl_time:.8f} seconds")

assert ampl_result["shipments"] == EXPECTED_POSITIVE_ARCS
assert ampl_result["total_cost"] == EXPECTED_COST
print("AMPL matches the textbook transportation plan.")


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 15609000
8 simplex iterations
1 branching nodes
=== TRANSPORTATION RESULTS WITH AMPL ===
Balanced problem check -> supply = 1450, demand = 1450
Solution               -> {'shipments': {('cali_plant', 'pasto'): 280, ('cali_plant', 'cali'): 170, ('cali_plant', 'manizales'): 150, ('medellin_plant', 'medellin'): 250, ('medellin_plant', 'cartagena'): 240, ('medellin_plant', 'manizales'): 60, ('medellin_plant', 'bogota'): 50, ('bogota_warehouse', 'bogota'): 250}, 'total_cost': 15609000}
Time                   -> 1.58658113 seconds
AMPL matches the textbook transportation plan.


## Variants in the Book

Section `2.7` does not add a student variation after the base transportation model, so this AMPL notebook closes with the balanced reference case only.
